In [39]:
import numpy as np 
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,StandardScaler
import torch

In [40]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')

In [41]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [42]:
df.isna().sum()

id                           0
diagnosis                    0
radius_mean                  0
texture_mean                 0
perimeter_mean               0
area_mean                    0
smoothness_mean              0
compactness_mean             0
concavity_mean               0
concave points_mean          0
symmetry_mean                0
fractal_dimension_mean       0
radius_se                    0
texture_se                   0
perimeter_se                 0
area_se                      0
smoothness_se                0
compactness_se               0
concavity_se                 0
concave points_se            0
symmetry_se                  0
fractal_dimension_se         0
radius_worst                 0
texture_worst                0
perimeter_worst              0
area_worst                   0
smoothness_worst             0
compactness_worst            0
concavity_worst              0
concave points_worst         0
symmetry_worst               0
fractal_dimension_worst      0
Unnamed:

In [43]:
try:
    df = df.drop(columns=['Unnamed: 32','id'])
except:
    pass

In [44]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
radius_mean,569.0,14.127292,3.524049,6.981000,11.700000,13.370000,15.780000,28.11000
texture_mean,569.0,19.289649,4.301036,9.710000,16.170000,18.840000,21.800000,39.28000
perimeter_mean,569.0,91.969033,24.298981,43.790000,75.170000,86.240000,104.100000,188.50000
area_mean,569.0,654.889104,351.914129,143.500000,420.300000,551.100000,782.700000,2501.00000
smoothness_mean,569.0,0.096360,0.014064,0.052630,0.086370,0.095870,0.105300,0.16340
compactness_mean,569.0,0.104341,0.052813,0.019380,0.064920,0.092630,0.130400,0.34540
concavity_mean,569.0,0.088799,0.079720,0.000000,0.029560,0.061540,0.130700,0.42680
concave points_mean,569.0,0.048919,0.038803,0.000000,0.020310,0.033500,0.074000,0.20120
symmetry_mean,569.0,0.181162,0.027414,0.106000,0.161900,0.179200,0.195700,0.30400
fractal_dimension_mean,569.0,0.062798,0.007060,0.049960,0.057700,0.061540,0.066120,0.09744


In [45]:
df.describe(exclude=np.number).T

,count,unique,top,freq
diagnosis,569,2,B,357


In [46]:
df['diagnosis'].value_counts()

diagnosis
B    357
M    212
Name: count, dtype: int64

In [47]:
X_train,X_test,y_train,y_test = train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=.2,random_state=42,stratify=df['diagnosis'])

In [48]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [49]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [50]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [51]:
X_train_tensor.shape

torch.Size([455, 30])

###### Define the model

In [52]:
class MySimpleNN():
    def __init__(self,X):
        self.weights = torch.rand(X.shape[1],1,dtype=torch.float64,requires_grad=True)
        self.bias = torch.zeros(1,dtype=torch.float64,requires_grad=True)

    def forward(self,X):
        z = torch.matmul(X,self.weights) + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred

    def loss_function(self,y,y_pred):
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred,epsilon,1-epsilon)
        loss = -((y*torch.log(y_pred)) + (1-y)*torch.log(1-y_pred)).mean()
        return loss


In [53]:
model = MySimpleNN(X_train_tensor)

for epoch in range(100):
    y_pred = model.forward(X_train_tensor)
    loss = model.loss_function(y_train_tensor,y_pred)
    loss.backward()
    with torch.no_grad():
        model.weights -= .1*model.weights.grad
        model.bias -= .1*model.bias.grad

    model.weights.grad.zero_()
    model.bias.grad.zero_()
    print(f"Epoch: {epoch+1}, Loss = {loss.item()}")

Epoch: 1, Loss = 3.337072549714656
Epoch: 2, Loss = 3.2088681287882936
Epoch: 3, Loss = 3.072626365385079
Epoch: 4, Loss = 2.935488746417322
Epoch: 5, Loss = 2.798835404552736
Epoch: 6, Loss = 2.6543115871077334
Epoch: 7, Loss = 2.506669458356197
Epoch: 8, Loss = 2.3600293529992933
Epoch: 9, Loss = 2.2129710996422043
Epoch: 10, Loss = 2.071874003507468
Epoch: 11, Loss = 1.932548732190482
Epoch: 12, Loss = 1.7957351861942952
Epoch: 13, Loss = 1.6619848286849137
Epoch: 14, Loss = 1.5345642659680556
Epoch: 15, Loss = 1.4174830160753094
Epoch: 16, Loss = 1.3079708151895433
Epoch: 17, Loss = 1.2124297799433434
Epoch: 18, Loss = 1.1310627556531163
Epoch: 19, Loss = 1.0633783087007673
Epoch: 20, Loss = 1.0081682913305279
Epoch: 21, Loss = 0.9636861793343152
Epoch: 22, Loss = 0.9279655976435294
Epoch: 23, Loss = 0.8991235857729141
Epoch: 24, Loss = 0.8755431404088247
Epoch: 25, Loss = 0.8559348399512999
Epoch: 26, Loss = 0.8393224960728751
Epoch: 27, Loss = 0.8249931709216879
Epoch: 28, Loss =

In [54]:
with torch.no_grad():
    y_pred = model.forward(X_train_tensor)
    y_pred = (y_pred>.5).float()
    accuracy = (y_pred == y_test_tensor).float().mean()
    print(accuracy.item())

0.6153846383094788


In [55]:
model.weights

tensor([[ 0.0341],
        [ 0.2098],
        [ 0.3580],
        [-0.3112],
        [ 0.0353],
        [-0.1151],
        [-0.2806],
        [ 0.2636],
        [ 0.1082],
        [ 0.0862],
        [ 0.0127],
        [ 0.2318],
        [-0.1508],
        [ 0.2777],
        [-0.0430],
        [-0.0741],
        [ 0.1034],
        [-0.1306],
        [-0.3702],
        [ 0.2432],
        [ 0.0073],
        [-0.2868],
        [-0.3445],
        [ 0.0518],
        [-0.1585],
        [-0.0862],
        [-0.1353],
        [ 0.3369],
        [ 0.3045],
        [-0.0668]], dtype=torch.float64, requires_grad=True)

In [56]:
model.bias

tensor([-0.4636], dtype=torch.float64, requires_grad=True)